# Text-to-Speech (TTS) — local speech generation on Apple Silicon

Pick a **model family** (Bark, Kitten-TTS, Qwen3-TTS, OuteTTS) and a **size** (SMALL / MEDIUM / LARGE), then run the
pipeline end to end:

1. Select the model from the `MODEL_NAME` registry (`APPLE → TTS → FAMILY → SIZE`)
2. Review every family and size: parameters, architecture, sample rate and memory
3. Load the model
4. Load the prompts (`prompts/tti_tts_ttt_prompts/tts_prompts_described.csv`: 10 prompts, each with a description of the voice to use)
5. Generate speech for every prompt, measuring speed, memory and energy, with per-prompt macmon capture
6. Listen to / save the audio
7. Check intelligibility (Whisper transcribes the generated audio) and the cost of the run
8. Compare SMALL / MEDIUM / LARGE of the selected family side by side
9. Combine every model run so far

All models run on **MLX** on the Apple GPU through [`mlx-audio`](https://github.com/Blaizzy/mlx-audio).

**MLX compatibility.** From the original registry, **Bark** is supported by mlx-audio, but **seamless-m4t-v2-large** is not:
it has no MLX conversion and mlx-audio has no implementation of its architecture, so it is not included.

## 1. Configuration — model registry

In [1]:
import os
import sys
from pathlib import Path

# Run from the project root, whether this notebook is opened from notebooks/ or from the root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'prompts').is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# Kitten-TTS converts text to phonemes with espeak-ng; use the copy bundled in the espeakng-loader package
import espeakng_loader
os.environ.setdefault('PHONEMIZER_ESPEAK_LIBRARY', espeakng_loader.get_library_path())
os.environ.setdefault('ESPEAK_DATA_PATH', espeakng_loader.get_data_path())

# APPLE -> TTS -> family -> size -> model
MODEL_NAME = {
    'APPLE': {
        'TTS': {
            'Bark': {                              # Suno Bark: GPT-style semantic -> coarse -> fine models + EnCodec
                'SMALL': 'bark-small',             # 420M
                'LARGE': 'bark',                   # 1.1B
            },
            'Kitten-TTS': {                        # tiny StyleTTS-style models
                'SMALL': 'kitten-tts-nano',        #  15M
                'MEDIUM': 'kitten-tts-micro',      #  36M
                'LARGE': 'kitten-tts-mini',        #  74M
            },
            'Qwen3-TTS': {                         # Qwen3 language model + 12 Hz speech tokenizer
                'SMALL': 'Qwen3-TTS-0.6B',         # 0.6B (1.1B incl. speech tokenizer)
                'LARGE': 'Qwen3-TTS-1.7B',         # 1.7B (2.1B incl. speech tokenizer)
            },
            'OuteTTS': {                           # Llama-3.2-1B language model + DAC audio codec
                'MEDIUM': 'OuteTTS-1.0-1B',        #  1.2B
            },
        },
    },
}

# Model -> MLX checkpoint (16-bit). 'models/...' = built locally on first use (section 3)
MLX_REPOS = {
    'bark-small':       'mlx-community/bark-small',
    # The mlx-community/bark upload stores attention as separate q/k/v weights and config fields mlx-audio doesn't read,
    # so a fixed local copy is built from it on first use
    'bark':             'models/bark',
    'kitten-tts-nano':  'mlx-community/kitten-tts-nano-0.8-fp16',
    'kitten-tts-micro': 'mlx-community/kitten-tts-micro-0.8-fp16',
    'kitten-tts-mini':  'mlx-community/kitten-tts-mini-0.8-fp16',
    'Qwen3-TTS-0.6B':   'mlx-community/Qwen3-TTS-12Hz-0.6B-CustomVoice-bf16',
    'Qwen3-TTS-1.7B':   'mlx-community/Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16',
    'OuteTTS-1.0-1B':   'mlx-community/Llama-OuteTTS-1.0-1B-fp16',
}

# Each prompt's voice follows the gender in its description (male / female); prompts without one use 'male'.
# Every size in a family uses the same voices, so sizes are compared like for like.
VOICES = {
    'Bark':       {'male': 'en_speaker_6',   'female': 'en_speaker_9'},     # Bark English presets
    'Kitten-TTS': {'male': 'expr-voice-5-m', 'female': 'expr-voice-5-f'},   # Kitten voices Leo / Kiki
    'Qwen3-TTS':  {'male': 'ryan',           'female': 'vivian'},           # Qwen3-TTS built-in speakers
    'OuteTTS':    {'male': 'default',        'female': 'default'},          # OuteTTS has one built-in speaker
}
DEFAULT_GENDER = 'male'
# Qwen3-TTS also follows the full description as a style instruction (tone, pace, pitch); the others cannot.
USE_DESCRIPTION_AS_INSTRUCTION = {'Qwen3-TTS'}

PLATFORM, TASK = 'APPLE', 'TTS'
# Environment variables TTS_FAMILY / TTS_SIZE / TTS_COMPARE override these, for running from the terminal
FAMILY = os.environ.get('TTS_FAMILY', 'Qwen3-TTS')   # 'Bark' | 'Kitten-TTS' | 'Qwen3-TTS' | 'OuteTTS'
SIZE = os.environ.get('TTS_SIZE', 'LARGE')           # 'SMALL' | 'MEDIUM' | 'LARGE' (only the sizes listed above)
RUN_FAMILY_COMPARISON = os.environ.get('TTS_COMPARE', '1') == '1'   # section 8: also run the family's other sizes

model_name = MODEL_NAME[PLATFORM][TASK][FAMILY][SIZE]
repo = MLX_REPOS[model_name]

# Prompt set: a CSV in prompts/tti_tts_ttt_prompts/ with columns id,prompt and optionally description. TTS_PROMPTS overrides it.
PROMPT_SET = os.environ.get('TTS_PROMPTS', 'tts_prompts_described')
INPUT_PATH = Path('prompts/tti_tts_ttt_prompts') / f'{PROMPT_SET}.csv'
OUTPUT_DIR = Path('outputs/tts') / FAMILY   # outputs/tts/<family>/<params>/<prompt set>/ (see out_dir, section 2)
WHISPER_REPO = 'mlx-community/whisper-large-v3-mlx'   # transcribes the generated audio to check intelligibility
SEED = 0                          # Bark, Qwen3-TTS and OuteTTS sample; a fixed seed makes runs repeatable
POWER_SAMPLE_MS = 100             # macmon sampling interval (its fastest is ~110 ms)
ELECTRICITY_USD_PER_KWH = 0.15    # your electricity price, for the $ per 1,000 audio hours figure

print(f'{PLATFORM} / {TASK} / {FAMILY} / {SIZE} -> {model_name} -> {repo}  (voices: {VOICES[FAMILY]})')
print(f'Prompts: {INPUT_PATH}')

APPLE / TTS / Qwen3-TTS / LARGE -> Qwen3-TTS-1.7B -> mlx-community/Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16  (voices: {'male': 'ryan', 'female': 'vivian'})
Prompts: prompts/tts_prompts_described.csv


## 2. Model families — sizes, architecture and memory

Parameter counts are summed from each checkpoint's weights (for Qwen3-TTS this includes its speech tokenizer).
All checkpoints run at 16-bit, so weights in memory ≈ the download size.

- **Bark** generates speech in three GPT-style stages (semantic tokens → coarse audio codes → fine audio codes), then
  decodes them with Meta's EnCodec. Expressive, but can add hesitations or ad-libs. It can only produce about 14 s of
  audio per call, so it is given one sentence at a time and the audio is joined.
- **Kitten-TTS** is a tiny StyleTTS-style model that turns phonemes directly into a waveform: very fast and small.
- **Qwen3-TTS** is a Qwen3 language model that writes speech tokens at 12 Hz, decoded by its own speech tokenizer.
- **OuteTTS** is a Llama-3.2-1B language model that writes DAC audio codes. Only the 1B size works in mlx-audio 0.5.7:
  the 0.6B (Qwen3-based) version runs but produces no audio codes.

**What an input costs for a local model.** There is **no per-character API fee ($0)**. What a prompt costs is
**time** (inference time and time to first audio), **memory** (peak GPU memory) and **energy** (section 7).

In [2]:
import mlx.core as mx
import pandas as pd

GB = 1e9
DEVICE = mx.device_info()

SPECS = pd.DataFrame([
    # model,             params,         architecture,                     sample_rate, languages,       download_gb
    ('bark-small',       420_400_000,    'GPT x3 + EnCodec',               24_000,      '13 languages',  1.61),
    ('bark',             1_143_600_000,  'GPT x3 + EnCodec',               24_000,      '13 languages',  4.42),
    ('kitten-tts-nano',  14_600_000,     'StyleTTS-style (phonemes)',      24_000,      'English',       0.06),
    ('kitten-tts-micro', 35_500_000,     'StyleTTS-style (phonemes)',      24_000,      'English',       0.13),
    ('kitten-tts-mini',  73_800_000,     'StyleTTS-style (phonemes)',      24_000,      'English',       0.29),
    ('Qwen3-TTS-0.6B',   1_076_300_000,  'Qwen3 LM + 12 Hz speech codec',  24_000,      '10 languages',  2.49),
    ('Qwen3-TTS-1.7B',   2_087_200_000,  'Qwen3 LM + 12 Hz speech codec',  24_000,      '10 languages',  4.52),
    ('OuteTTS-1.0-1B',   1_248_400_000,  'Llama-3.2 LM + DAC codec',       24_000,      '20+ languages', 2.50),
], columns=['model', 'params', 'architecture', 'sample_rate', 'languages', 'download_gb']).set_index('model')

# Output subfolder per model, named by its parameter count: outputs/tts/<family>/<params>/<prompt set>/
PARAM_FOLDER = {'bark-small': '420M', 'bark': '1.1B', 'kitten-tts-nano': '15M', 'kitten-tts-micro': '36M',
                'kitten-tts-mini': '74M', 'Qwen3-TTS-0.6B': '0.6B', 'Qwen3-TTS-1.7B': '1.7B', 'OuteTTS-1.0-1B': '1B'}

def family_of(name):
    return next(f for f, sizes in MODEL_NAME[PLATFORM][TASK].items() if name in sizes.values())

def size_of(name):
    return next(sz for sizes in MODEL_NAME[PLATFORM][TASK].values() for sz, n in sizes.items() if n == name)

def out_dir(name):
    d = OUTPUT_DIR / PARAM_FOLDER[name] / PROMPT_SET
    d.mkdir(parents=True, exist_ok=True)
    return d

rows = []
for family, sizes in MODEL_NAME[PLATFORM][TASK].items():
    for size, name in sizes.items():
        s = SPECS.loc[name]
        rows.append({
            '': '▶' if name == model_name else '',
            'Family': family, 'Size': size, 'Model': name,
            'Params (M)': f'{s.params / 1e6:,.0f}',
            'Architecture': s.architecture,
            'Sample rate (Hz)': f'{s.sample_rate:,}',
            'Languages': s.languages,
            'Voices (male / female)': ' / '.join(VOICES[family].values()),
            'Download (GB)': f'{s.download_gb:.2f}',
            'MLX checkpoint': MLX_REPOS[name],
            'API cost': '$0 (local)',
        })
print(f"{DEVICE['device_name']}: {DEVICE['memory_size'] / GB:.0f} GB unified memory.  ▶ = selected model")
pd.DataFrame(rows).style.hide(axis='index')

Apple M5 Max: 52 GB unified memory.  ▶ = selected model


,Family,Size,Model,Params (M),Architecture,Sample rate (Hz),Languages,Voices (male / female),Download (GB),MLX checkpoint,API cost
,Bark,SMALL,bark-small,420,GPT x3 + EnCodec,"24,000",13 languages,en_speaker_6 / en_speaker_9,1.61,mlx-community/bark-small,$0 (local)
,Bark,LARGE,bark,"1,144",GPT x3 + EnCodec,"24,000",13 languages,en_speaker_6 / en_speaker_9,4.42,models/bark,$0 (local)
,Kitten-TTS,SMALL,kitten-tts-nano,15,StyleTTS-style (phonemes),"24,000",English,expr-voice-5-m / expr-voice-5-f,0.06,mlx-community/kitten-tts-nano-0.8-fp16,$0 (local)
,Kitten-TTS,MEDIUM,kitten-tts-micro,36,StyleTTS-style (phonemes),"24,000",English,expr-voice-5-m / expr-voice-5-f,0.13,mlx-community/kitten-tts-micro-0.8-fp16,$0 (local)
,Kitten-TTS,LARGE,kitten-tts-mini,74,StyleTTS-style (phonemes),"24,000",English,expr-voice-5-m / expr-voice-5-f,0.29,mlx-community/kitten-tts-mini-0.8-fp16,$0 (local)
,Qwen3-TTS,SMALL,Qwen3-TTS-0.6B,"1,076",Qwen3 LM + 12 Hz speech codec,"24,000",10 languages,ryan / vivian,2.49,mlx-community/Qwen3-TTS-12Hz-0.6B-CustomVoice-bf16,$0 (local)
▶,Qwen3-TTS,LARGE,Qwen3-TTS-1.7B,"2,087",Qwen3 LM + 12 Hz speech codec,"24,000",10 languages,ryan / vivian,4.52,mlx-community/Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16,$0 (local)
,OuteTTS,MEDIUM,OuteTTS-1.0-1B,"1,248",Llama-3.2 LM + DAC codec,"24,000",20+ languages,default / default,2.50,mlx-community/Llama-OuteTTS-1.0-1B-fp16,$0 (local)


## 3. Load the model

**Bark (large)** is the one model built locally. The `mlx-community/bark` upload is MLX-format, but it stores each
attention layer's query/key/value weights separately (mlx-audio expects them combined, as in bark-small), keeps
the fine-acoustics weights in a second file, and uses config field names mlx-audio doesn't read. On first use the
notebook builds `models/bark/` from it: q/k/v combined in Bark's order, the causal attention masks added, the config
fields filled in and the tokenizer copied from bark-small (both sizes share it). The weight values are unchanged.

In [3]:
import glob
import json
import re
import shutil
import time
import numpy as np
from huggingface_hub import hf_hub_download, snapshot_download
from mlx.utils import tree_flatten
from mlx_audio.tts.utils import load_model

def build_local_bark(dst='models/bark'):
    """Build a loadable Bark (large) from mlx-community/bark (see the note above)."""
    print('Building models/bark from mlx-community/bark, one-time step...')
    src = Path(snapshot_download('mlx-community/bark', token=False,
                                 allow_patterns=['*.safetensors', '*.json', 'README.md', 'v2/en_*.npz']))
    small = Path(snapshot_download('mlx-community/bark-small', token=False,
                                   allow_patterns=['tokenizer*', 'special_tokens_map.json', 'vocab.txt']))
    w = {}
    for f in ('model.safetensors', 'fine_acoustics.safetensors'):
        w.update(mx.load(str(src / f)))
    out = {}
    for k, v in w.items():
        if k.endswith('attn.q_proj.weight'):
            base = k[:-len('q_proj.weight')]
            out[base + 'att_proj.weight'] = mx.concatenate(
                [w[base + 'q_proj.weight'], w[base + 'k_proj.weight'], w[base + 'v_proj.weight']], axis=0)
        elif not k.endswith(('attn.k_proj.weight', 'attn.v_proj.weight')):
            out[k] = v
    cfg = json.loads((src / 'config.json').read_text())
    for part in ('semantic_config', 'coarse_acoustics_config', 'fine_acoustics_config'):
        c = cfg[part]
        c.update(n_layer=c['num_layers'], n_head=c['num_heads'], n_embd=c['hidden_size'])
    cfg.update(model_size='large', n_layer=24, n_head=16, n_embd=1024)
    block = cfg['semantic_config']['block_size']
    mask = mx.tril(mx.ones((block, block))).astype(mx.bool_).reshape(1, 1, block, block)
    for part in ('semantic', 'coarse_acoustics'):   # causal masks for the two causal stages
        for i in range(cfg['semantic_config']['n_layer']):
            out[f'{part}.layers.{i}.attn.bias'] = mask
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    mx.eval(out)
    mx.save_safetensors(str(dst / 'model.tmp.safetensors'), out)
    os.replace(dst / 'model.tmp.safetensors', dst / 'model.safetensors')
    (dst / 'config.json').write_text(json.dumps(cfg, indent=2))
    shutil.copy(src / 'generation_config.json', dst / 'generation_config.json')
    for f in ('tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt'):
        shutil.copy(small / f, dst / f)

def gender_of(description):
    d = str(description or '').lower()
    if re.search(r'\bfemale\b|\bwoman\b', d):
        return 'female'
    if re.search(r'\bmale\b|\bman\b', d):
        return 'male'
    return DEFAULT_GENDER

def voice_kwargs(name, description=None):
    """generate() arguments that select the voice for one prompt."""
    family = family_of(name)
    voice = VOICES[family][gender_of(description)]
    if family == 'Bark':
        if name == 'bark-small':   # bark-small ships this preset as three .npy prompts
            return {'voice': {k: np.load(hf_hub_download('mlx-community/bark-small', token=False,
                                                         filename=f'speaker_embeddings/{voice}_{k}.npy'))
                              for k in ('semantic_prompt', 'coarse_prompt', 'fine_prompt')}}
        return {'voice': hf_hub_download('mlx-community/bark', f'v2/{voice}.npz', token=False)}
    if family == 'Kitten-TTS':
        return {'voice': voice}
    if family == 'Qwen3-TTS':
        kw = {'voice': voice, 'lang_code': 'english'}
        if family in USE_DESCRIPTION_AS_INSTRUCTION and isinstance(description, str) and description:
            kw['instruct'] = description
        return kw
    return {'max_tokens': 4096}    # OuteTTS: default speaker; its default 1,200-token cap allows only ~8 s of audio

def load_tts_model(name):
    repo = MLX_REPOS[name]
    if name == 'bark' and not Path(repo, 'model.safetensors').exists():
        build_local_bark(repo)
    t0 = time.perf_counter()
    model = load_model(repo)
    mx.eval(model.parameters())
    return model, time.perf_counter() - t0

model, load_seconds = load_tts_model(model_name)
spec = SPECS.loc[model_name]
print(f'Loaded {repo} in {load_seconds:.1f}s on {mx.default_device()}')
print(f'Parameters: {spec.params / 1e6:,.0f}M  |  {spec.architecture}  |  {spec.sample_rate:,} Hz output')
print(f'Weights in memory: {mx.get_active_memory() / GB:.2f} GB')

/Users/genailab/mlx-projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 2159.88it/s]

[transformers] You are using a model of type `qwen3_tts` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


  Initialized encoder codebooks
Loaded speech tokenizer from /Users/genailab/.cache/huggingface/hub/models--mlx-community--Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16/snapshots/52f4770fd9726457eae3d3b6aa92047a25a10776/speech_tokenizer
Loaded mlx-community/Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16 in 2.8s on Device(gpu, 0)
Parameters: 2,087M  |  Qwen3 LM + 12 Hz speech codec  |  24,000 Hz output
Weights in memory: 4.58 GB


## 4. Load the prompts

Each prompt comes with a description of the voice that should read it. How each family uses the description:

- **Qwen3-TTS** follows the whole description as a style instruction (`instruct=`), and the speaker is chosen by gender
  (Ryan for male, Vivian for female)
- **Bark** and **Kitten-TTS** cannot follow free-text descriptions, so only the gender is used to pick a male or female preset
- **OuteTTS** has a single built-in speaker, so the description is not used

Descriptions without a gender (e.g. "a neutral robotic voice, genderless") use each family's default voice (male).

In [4]:
PROMPTS = pd.read_csv(INPUT_PATH, dtype={'id': str})
PROMPTS['id'] = PROMPTS['id'].str.zfill(2)
PROMPTS['words'] = PROMPTS.prompt.str.split().str.len()
PROMPTS['characters'] = PROMPTS.prompt.str.len()
if 'description' not in PROMPTS:
    PROMPTS['description'] = ''
PROMPTS['description'] = PROMPTS['description'].fillna('')
PROMPTS['gender'] = PROMPTS.description.map(gender_of)
print(f'Prompts: {INPUT_PATH} ({len(PROMPTS)} prompts, {PROMPTS.words.sum()} words, {PROMPTS.characters.sum():,} characters)')
pd.DataFrame({
    'id': PROMPTS.id,
    'words': PROMPTS.words,
    'characters': PROMPTS.characters,
    'voice gender': PROMPTS.gender,
    'prompt': PROMPTS.prompt.map(lambda t: t if len(t) <= 70 else t[:67] + '...'),
    'description': PROMPTS.description.map(lambda t: t if len(t) <= 70 else t[:67] + '...'),
}).style.hide(axis='index')

Prompts: prompts/tts_prompts_described.csv (10 prompts, 271 words, 1,635 characters)


id,words,characters,voice gender,prompt,description
01,34,227,male,"Welcome to our annual conference on AI advancements! Today, we gath...",A formal male speaker delivers the message in a confident and profe...
02,22,149,female,Congratulations! You've won the grand prize! It’s your moment to sh...,"A female speaker with a cheerful and enthusiastic tone, speaking wi..."
03,28,157,male,Please make your way to the nearest exit in an orderly manner. For ...,"A neutral, clear, and calm male speaker delivering instructions in ..."
04,29,155,female,It’s going to be a beautiful day with sunshine and clear skies! Ste...,"A female speaker with a warm and optimistic tone, delivering the we..."
05,25,150,male,Our sale ends tonight at midnight—don’t miss out on these exclusive...,"A dynamic male speaker with a persuasive and engaging tone, speakin..."
06,26,155,female,Can you believe it? We made it to the finals! After all our hard wo...,"A female speaker with a surprised and joyful tone, delivering the m..."
07,31,189,male,"In 1492, Christopher Columbus set sail across the Atlantic, embarki...","A male narrator with a steady and informative tone, delivering the ..."
08,28,155,female,Your food will be served shortly. Thank you for your patience as we...,"A soft-spoken female voice, polite and calm, with a soothing tone s..."
09,23,143,male,Error: The operation could not be completed. Please try again later...,"A neutral robotic voice, genderless, speaking in a monotone and pre..."
10,25,155,male,"Are you ready for an adventure of a lifetime? Pack your bags, gear ...","A male speaker with an enthusiastic and adventurous tone, slightly ..."


## 5. Generate speech

Each prompt is synthesised separately with the voice chosen for it (section 4) and a fixed random seed.

**Power monitoring.** `macmon` (no sudo needed) runs in the background for the whole notebook, sampling the chip's
power sensors about every 110 ms. Each prompt's energy is the power integrated over the time it took to generate.
If `macmon` isn't installed, the power metrics are left empty and everything else runs. Readings are Apple's
on-chip estimates, not a wall-meter measurement.

In [5]:
import atexit
import json
import shutil
import subprocess
import threading

class PowerMonitor:
    """Runs `macmon pipe` in the background and keeps every sample.

    Each sample is the average over the interval since the previous one, so the energy in any time
    window is the sum of power x (overlap of that interval with the window).
    """
    def __init__(self, interval_ms):
        self.samples = []  # (time.perf_counter(), macmon sample dict)
        self.available = shutil.which('macmon') is not None
        if not self.available:
            print('macmon not found: power and energy metrics will be empty (install: brew install macmon)')
            return
        self.proc = subprocess.Popen(['macmon', 'pipe', '-i', str(interval_ms)],
                                     stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        atexit.register(self.proc.terminate)
        threading.Thread(target=self._read, daemon=True).start()
        t_start = time.perf_counter()
        while len(self.samples) < 2 and time.perf_counter() - t_start < 5:  # macmon takes ~0.7 s to start
            time.sleep(0.05)
        self.available = len(self.samples) >= 2

    def _read(self):
        for line in self.proc.stdout:
            try:
                self.samples.append((time.perf_counter(), json.loads(line)))
            except json.JSONDecodeError:
                pass

    def wait_past(self, t, timeout=2.0):
        """Block until a sample covering time t has arrived."""
        deadline = time.perf_counter() + timeout
        while self.samples[-1][0] < t and time.perf_counter() < deadline:
            time.sleep(0.02)

    def window(self, a, b):
        """Samples overlapping [a, b], as (seconds of overlap, sample)."""
        return [(min(t, b) - max(t_prev, a), s)
                for (t_prev, _), (t, s) in zip(self.samples, self.samples[1:])
                if min(t, b) - max(t_prev, a) > 0]

def soc_power(s):
    return s['all_power'] + s['ram_power']  # CPU + GPU + Neural Engine + memory, in watts

def integrate(win, fn):
    return sum(overlap * fn(s) for overlap, s in win)

POWER_KEYS = ['gpu_util_avg_pct', 'gpu_power_avg_w', 'gpu_power_peak_w', 'soc_power_avg_w', 'gpu_temp_max_c',
              'energy_j', 'energy_net_j', 'power_samples']

def summarize_power(t0, t_end):
    if not POWER.available:
        return dict.fromkeys(POWER_KEYS)
    POWER.wait_past(t_end)
    run = POWER.window(t0, t_end)
    samples = [s for _, s in run]
    dur = t_end - t0
    e_soc = integrate(run, soc_power)
    return {
        'gpu_util_avg_pct': round(100 * integrate(run, lambda s: s['gpu_active_ratio']) / dur, 1),
        'gpu_power_avg_w': round(integrate(run, lambda s: s['gpu_power']) / dur, 1),
        'gpu_power_peak_w': round(max(s['gpu_power'] for s in samples), 1),
        'soc_power_avg_w': round(e_soc / dur, 1),
        'gpu_temp_max_c': round(max(s['temp']['gpu_temp_avg'] for s in samples), 1),
        'energy_j': round(e_soc, 2),
        'energy_net_j': round(e_soc - IDLE_SOC_W * dur, 2),
        'power_samples': len(run),
    }

POWER = PowerMonitor(POWER_SAMPLE_MS)
if POWER.available:
    time.sleep(1.0)  # idle baseline, measured with the model loaded but not running
    a, b = POWER.samples[0][0], POWER.samples[-1][0]
    IDLE_SOC_W = integrate(POWER.window(a, b), soc_power) / (b - a)
    print(f'Power monitor running (macmon, ~{POWER_SAMPLE_MS} ms samples). Idle chip power: {IDLE_SOC_W:.2f} W')

def cpu_clusters():
    """CPU cluster names and core counts from macOS (e.g. M5 Max: 6 'Super' + 12 'Performance' cores)."""
    def sysctl(key):
        return subprocess.run(['sysctl', '-n', key], capture_output=True, text=True).stdout.strip()
    return [(sysctl(f'hw.perflevel{i}.name') or f'level{i}', int(sysctl(f'hw.perflevel{i}.logicalcpu') or 0)) for i in (0, 1)]

CPU_CLUSTERS = cpu_clusters()
HARDWARE = {}   # model name -> hardware_metrics() for its whole run

def hardware_metrics(t0, t1):
    """GPU, CPU, Neural Engine (NPU), memory and whole-chip activity over [t0, t1], from the macmon samples.

    macmon calls the two CPU clusters 'pcpu' (perflevel0) and 'ecpu' (perflevel1); they are reported here
    under the names macOS gives them.
    """
    if not POWER.available:
        return None
    POWER.wait_past(t1)
    win = POWER.window(t0, t1)
    samples = [s for _, s in win]
    dur = t1 - t0
    avg = lambda fn: integrate(win, fn) / dur
    peak = lambda fn: max(fn(s) for s in samples)
    energy = lambda fn: round(integrate(win, fn), 2)
    pct = lambda x: round(100 * x, 1)
    (c0_name, c0_cores), (c1_name, c1_cores) = CPU_CLUSTERS
    c0, c1 = c0_name.lower(), c1_name.lower()
    ane = lambda s: s['ane_power']
    return {
        'gpu': {
            'avg_utilization_pct': pct(avg(lambda s: s['gpu_active_ratio'])),
            'peak_utilization_pct': pct(peak(lambda s: s['gpu_active_ratio'])),
            'avg_frequency_mhz': round(avg(lambda s: s['gpu_freq_mhz'])),
            'max_frequency_mhz': round(peak(lambda s: s['gpu_freq_mhz'])),
            'avg_power_watts': round(avg(lambda s: s['gpu_power']), 1),
            'peak_power_watts': round(peak(lambda s: s['gpu_power']), 1),
            'energy_joules': energy(lambda s: s['gpu_power']),
            'avg_temperature_c': round(avg(lambda s: s['temp']['gpu_temp_avg']), 1),
            'max_temperature_c': round(peak(lambda s: s['temp']['gpu_temp_avg']), 1),
        },
        'cpu': {
            'avg_utilization_pct': pct(avg(lambda s: s['cpu_active_ratio'])),
            'peak_utilization_pct': pct(peak(lambda s: s['cpu_active_ratio'])),
            f'{c0}_cores': c0_cores,
            f'{c0}_cores_avg_utilization_pct': pct(avg(lambda s: s['pcpu_active_ratio'])),
            f'{c0}_cores_avg_frequency_mhz': round(avg(lambda s: s['pcpu_freq_mhz'])),
            f'{c1}_cores': c1_cores,
            f'{c1}_cores_avg_utilization_pct': pct(avg(lambda s: s['ecpu_active_ratio'])),
            f'{c1}_cores_avg_frequency_mhz': round(avg(lambda s: s['ecpu_freq_mhz'])),
            'avg_power_watts': round(avg(lambda s: s['cpu_power']), 1),
            'peak_power_watts': round(peak(lambda s: s['cpu_power']), 1),
            'energy_joules': energy(lambda s: s['cpu_power']),
            'avg_temperature_c': round(avg(lambda s: s['temp']['cpu_temp_avg']), 1),
            'max_temperature_c': round(peak(lambda s: s['temp']['cpu_temp_avg']), 1),
        },
        'npu': {
            'name': 'Apple Neural Engine (ANE)',
            'used': peak(ane) > 0.05,
            'avg_power_watts': round(avg(ane), 2),
            'peak_power_watts': round(peak(ane), 2),
            'energy_joules': energy(ane),
            'note': 'MLX runs models on the GPU, so the Neural Engine stays idle (about 0 W) during these runs.',
        },
        'system_memory': {
            'avg_power_watts': round(avg(lambda s: s['ram_power']), 1),
            'energy_joules': energy(lambda s: s['ram_power']),
            'ram_total_gb': round(samples[0]['memory']['ram_total'] / 1e9, 1),
            'avg_ram_used_gb': round(avg(lambda s: s['memory']['ram_usage']) / 1e9, 1),
            'peak_ram_used_gb': round(peak(lambda s: s['memory']['ram_usage']) / 1e9, 1),
            'peak_swap_used_gb': round(peak(lambda s: s['memory']['swap_usage']) / 1e9, 2),
            'note': 'RAM used is for the whole Mac, including other apps, not just this model.',
        },
        'whole_chip': {
            'avg_power_watts': round(avg(soc_power), 1),
            'peak_power_watts': round(peak(soc_power), 1),
            'energy_joules': energy(soc_power),
            'idle_power_watts': round(IDLE_SOC_W, 2),
            'note': 'CPU + GPU + Neural Engine + memory.',
        },
        'measured_seconds': round(dur, 2),
        'power_samples': len(win),
    }

Power monitor running (macmon, ~100 ms samples). Idle chip power: 0.29 W


### Per-prompt metrics analysis

Every prompt is wrapped in `metrics_monitor_start()` / `metrics_monitor_stop()` from `macmon_metrics/metrics_monitor.py`,
which records a macmon time series for that prompt. Results go to a separate folder,
`metrics_analysis/tts/<family>/<params>/<prompt set>/`:

- `System_Level_Metrics/<prompt id>_system_metrics.csv`: CPU / GPU / NPU utilisation, power and temperature, RAM used
  and total system power, sampled every 100 ms while that prompt was synthesised
- `Model_Level_Metrics/<model>_prompt_metrics.csv`: one row per prompt (columns explained in section 7)

How the speech-model numbers are defined:

- **Time to first audio** = time until the first piece of audio is returned. These models return audio a sentence
  (or a whole prompt) at a time rather than streaming it, so this is close to the inference time for short prompts.
- **Inference rate** = seconds of audio produced per second of compute (above 1 = faster than real time).
- **Word error rate** = how far Whisper's transcript of the generated audio is from the prompt (lower = clearer speech).

Section 9 combines every model's `Model_Level_Metrics` CSV into `metrics_analysis/tts/all_models_prompt_metrics_<prompt set>.csv`.

In [6]:
import re
import soundfile as sf
import mlx_whisper
from mlx_whisper.transcribe import ModelHolder
import macmon_metrics.metrics_monitor as mm
from macmon_metrics.metrics_monitor import metrics_monitor_start, metrics_monitor_stop, setup_output_directory

METRICS_ROOT = Path('metrics_analysis/tts')

def analysis_dirs(name):
    """(System_Level_Metrics, Model_Level_Metrics) folders for one model."""
    return setup_output_directory(str(METRICS_ROOT / family_of(name) / PARAM_FOLDER[name] / PROMPT_SET))

# Start the macmon tracker now: it takes about a second to produce its first sample, and starting it here means
# every prompt's capture has samples from its first moment.
if mm._tracker is None:
    mm._tracker = mm.MacMonTracker(binary_path=mm._resolve_macmon_binary())
    mm._tracker.start()
    time.sleep(1.5)

def split_sentences(text):
    """Bark can only produce about 14 s of audio per call, so it is given one sentence at a time."""
    parts = re.split(r'(?<=[.!?])\s*(?=[A-Z(])', text.strip())
    return [p for p in parts if p.strip()]

def synthesize(model, name, text, voice):
    """Generate speech for one prompt. Returns the waveform and timings."""
    mx.random.seed(SEED)
    mx.reset_peak_memory()
    pieces = split_sentences(text) if family_of(name) == 'Bark' else [text]
    chunks, t_first, sample_rate = [], None, None
    t0 = time.perf_counter()
    for piece in pieces:
        for r in model.generate(text=piece, **voice):
            if t_first is None:
                t_first = time.perf_counter()
            chunks.append(np.array(r.audio, dtype=np.float32).reshape(-1))
            sample_rate = r.sample_rate
    t_end = time.perf_counter()
    audio = np.concatenate(chunks) if chunks else np.zeros(0, dtype=np.float32)
    return {
        'audio': audio,
        'sample_rate': sample_rate,
        'audio_s': len(audio) / sample_rate if sample_rate else 0.0,
        'time_s': t_end - t0,
        'ttfa_s': (t_first or t_end) - t0,
        'peak_gb': mx.get_peak_memory() / GB,
        'power': summarize_power(t0, t_end),
    }

def warm_up(model, name):
    """Synthesise the first prompt once, untimed, so GPU kernels are compiled before anything is measured."""
    synthesize(model, name, PROMPTS.prompt.iloc[0], voice_kwargs(name, PROMPTS.description.iloc[0]))

def run_prompt_set(model, name):
    warm_up(model, name)
    sys_dir, model_dir = analysis_dirs(name)
    runs = []
    t_start = time.perf_counter()
    for row in PROMPTS.itertuples(index=False):
        metrics_monitor_start(idx=row.id, metrics_folder=sys_dir)
        kw = voice_kwargs(name, row.description)
        g = synthesize(model, name, row.prompt, kw)
        metrics_monitor_stop()
        wav = out_dir(name) / f'{name}_audio_{row.id}.wav'
        sf.write(wav, g['audio'], g['sample_rate'], subtype='PCM_16')
        g.update(id=row.id, prompt=row.prompt, description=row.description, words=row.words, characters=row.characters,
                 audio_file=str(wav), voice=VOICES[family_of(name)][row.gender], instructed='instruct' in kw)
        del g['audio']
        runs.append(g)
        print(f"  [{row.id}] {row.words:>3} words -> {g['audio_s']:5.1f}s audio in {g['time_s']:5.2f}s "
              f"({g['audio_s'] / g['time_s']:5.2f} audio s/s, first audio {1000 * g['ttfa_s']:6.0f} ms)")
    HARDWARE[name] = hardware_metrics(t_start, time.perf_counter())
    return runs, model_dir

print(f'Generating speech for {len(PROMPTS)} prompts with {model_name}...')
runs, model_dir = run_prompt_set(model, model_name)
print(f"Done: {sum(g['audio_s'] for g in runs):.0f}s of audio in {sum(g['time_s'] for g in runs):.1f}s")

Generating speech for 10 prompts with Qwen3-TTS-1.7B...


  [01]  34 words ->  18.2s audio in  4.30s ( 4.24 audio s/s, first audio   4288 ms)


  [02]  22 words ->  10.9s audio in  2.60s ( 4.19 audio s/s, first audio   2587 ms)


  [03]  28 words ->  11.0s audio in  2.62s ( 4.22 audio s/s, first audio   2608 ms)


  [04]  29 words ->  11.8s audio in  2.79s ( 4.22 audio s/s, first audio   2777 ms)


...

  [06]  26 words ->  11.8s audio in  2.78s ( 4.23 audio s/s, first audio   2774 ms)


  [07]  31 words ->  17.1s audio in  4.04s ( 4.24 audio s/s, first audio   4025 ms)


  [08]  28 words ->  11.5s audio in  2.77s ( 4.16 audio s/s, first audio   2764 ms)


  [09]  23 words ->  13.8s audio in  3.26s ( 4.24 audio s/s, first audio   3251 ms)


  [10]  25 words ->   9.9s audio in  2.37s ( 4.19 audio s/s, first audio   2357 ms)
Done: 126s of audio in 30.0s


## 6. Listen to the audio

One WAV file per prompt: `outputs/tts/<family>/<params>/<prompt set>/<model>_audio_<id>.wav`.

In [7]:
from IPython.display import Audio, display

print(f'Saved {len(runs)} WAV files to {out_dir(model_name)}/')
print(f"Prompt {runs[0]['id']}: {runs[0]['prompt']}")
display(Audio(runs[0]['audio_file']))

Saved 10 WAV files to outputs/tts/Qwen3-TTS/1.7B/tts_prompts_described/
Prompt 01: Welcome to our annual conference on AI advancements! Today, we gather here to celebrate the latest breakthroughs in artificial intelligence. Let us explore innovative technologies and pave the way for a smarter future together!


## 7. Intelligibility and cost of this run

**Intelligibility check.** Whisper (`whisper-large-v3`) transcribes every generated clip, and the transcript is compared
with the prompt as a **word error rate** (WER: substituted + inserted + deleted words ÷ prompt words, after normalising
case, punctuation and number words, so "five hundred" matches "500"). This is an automatic proxy for how clearly the
model spoke; it does not judge naturalness. Whisper runs after generation, outside the timed and measured windows.

**Per-prompt CSV columns** (`Model_Level_Metrics/<model>_prompt_metrics.csv`):

| Column | Meaning |
|---|---|
| `prompt_id`, `model`, `family`, `size`, `voice` | which prompt, model and voice |
| `description_used` | how the voice description was applied: `full description`, `gender only` or `no` |
| `input_words`, `input_characters` | length of the prompt text |
| `output_audio (s)`, `sample_rate (Hz)` | length and sample rate of the generated speech |
| `time_to_first_audio (ms)` | time until the first audio was returned |
| `inference_rate (audio s/s)`, `inference_time (s)` | seconds of audio per second of compute, and total generation time |
| `word_error_rate (%)` | Whisper round-trip error (lower = clearer) |
| `audio_file`, `prompt`, `description`, `transcript` | the WAV file, the text spoken, the requested voice, and what Whisper heard |

In [8]:
import gc
import re
import difflib

UNITS = {w: i for i, w in enumerate(
    'zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen '
    'fifteen sixteen seventeen eighteen nineteen'.split())}
TENS = {w: 10 * i for i, w in enumerate('twenty thirty forty fifty sixty seventy eighty ninety'.split(), 2)}

def words_to_digits(words):
    out, num, active = [], 0, False
    for w in words:
        if w in UNITS:
            num += UNITS[w]; active = True
        elif w in TENS:
            num += TENS[w]; active = True
        elif w == 'hundred' and active:
            num *= 100
        elif w == 'thousand' and active:
            num *= 1000
        elif w == 'and' and active:
            continue
        else:
            if active:
                out.append(str(num)); num, active = 0, False
            out.append(w)
    if active:
        out.append(str(num))
    return out

def normalize(text):
    text = text.lower().replace('’', "'").replace('-', ' ')
    text = re.sub(r'\.(?!\d)', '', text)                       # U.K. -> uk, keep decimals
    text = re.sub(r"[^a-z0-9' ]+", ' ', text)
    text = re.sub(r'(\d)([a-z])\b', r'\1 \2', text)            # 172n -> 172 n
    return words_to_digits(text.split())

def edit_counts(ref, hyp):
    s = d = i = 0
    for op, a0, a1, b0, b1 in difflib.SequenceMatcher(a=ref, b=hyp, autojunk=False).get_opcodes():
        if op == 'replace':
            s += min(a1 - a0, b1 - b0); d += max(0, (a1 - a0) - (b1 - b0)); i += max(0, (b1 - b0) - (a1 - a0))
        elif op == 'delete':
            d += a1 - a0
        elif op == 'insert':
            i += b1 - b0
    return s, d, i

def whisper_check(runs):
    """Transcribe each generated clip with Whisper and score it against the prompt."""
    for g in runs:
        g['transcript'] = mlx_whisper.transcribe(g['audio_file'], path_or_hf_repo=WHISPER_REPO, language='en',
                                                 verbose=None, condition_on_previous_text=False)['text'].strip()
        ref, hyp = normalize(g['prompt']), normalize(g['transcript'])
        g['ref_words'], g['word_errors'] = len(ref), sum(edit_counts(ref, hyp))
        g['wer_pct'] = round(100 * g['word_errors'] / len(ref), 2)
    # Unload Whisper so it doesn't stay in GPU memory (and in the next model's peak memory)
    ModelHolder.model, ModelHolder.model_path = None, None
    gc.collect()
    mx.clear_cache()

def save_prompt_metrics(name, runs, folder):
    """Model_Level_Metrics CSV: one row per prompt."""
    rows = [{
        'prompt_id': g['id'],
        'model': name,
        'family': family_of(name),
        'size': size_of(name),
        'voice': g['voice'],
        'description_used': 'full description' if g['instructed'] else 'gender only' if family_of(name) != 'OuteTTS' else 'no',
        'input_words': g['words'],
        'input_characters': g['characters'],
        'output_audio (s)': round(g['audio_s'], 2),
        'sample_rate (Hz)': g['sample_rate'],
        'time_to_first_audio (ms)': round(1000 * g['ttfa_s'], 1),
        'inference_rate (audio s/s)': round(g['audio_s'] / g['time_s'], 2),
        'inference_time (s)': round(g['time_s'], 3),
        'word_error_rate (%)': g['wer_pct'],
        'audio_file': g['audio_file'],
        'prompt': g['prompt'],
        'description': g['description'],
        'transcript': g['transcript'],
    } for g in runs]
    path = Path(folder) / f'{name}_prompt_metrics.csv'
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

def simple_metrics(name, runs, load_s):
    """A flat, readable summary of one model's run: plain names, units in every key."""
    s = SPECS.loc[name]
    n = len(runs)
    audio_s = sum(g['audio_s'] for g in runs)
    time_s = sum(g['time_s'] for g in runs)
    durations = [g['audio_s'] for g in runs]
    errors, ref_words = sum(g['word_errors'] for g in runs), sum(g['ref_words'] for g in runs)
    worst = max(runs, key=lambda g: g['wer_pct'])
    measured = runs[0]['power']['energy_j'] is not None
    energy = sum(g['power']['energy_j'] for g in runs) if measured else None
    return {
        'model': name, 'family': family_of(name), 'size': size_of(name), 'repo': MLX_REPOS[name],
        'parameters_million': round(s.params / 1e6), 'architecture': s.architecture, 'precision': '16-bit',
        'voices': VOICES[family_of(name)], 'prompt_set': PROMPT_SET, 'prompts_run': n,
        'input': {'total_words': int(sum(g['words'] for g in runs)), 'total_characters': int(sum(g['characters'] for g in runs))},
        'audio_output': {
            'total_seconds': round(audio_s, 1), 'avg_seconds_per_prompt': round(audio_s / n, 1),
            'shortest_seconds': round(min(durations), 1), 'longest_seconds': round(max(durations), 1),
            'sample_rate_hz': runs[0]['sample_rate'],
        },
        'intelligibility': {
            'word_error_rate_pct': round(100 * errors / ref_words, 2),
            'word_accuracy_pct': round(100 - 100 * errors / ref_words, 2),
            'word_errors': errors, 'prompt_words': ref_words,
            'perfect_prompts': sum(1 for g in runs if g['wer_pct'] == 0),
            'worst_prompt_id': worst['id'], 'worst_prompt_word_error_rate_pct': worst['wer_pct'],
            'checked_with': WHISPER_REPO,
        },
        'speed': {
            'model_load_seconds': round(load_s, 1),
            'total_inference_seconds': round(time_s, 2),
            'avg_seconds_per_prompt': round(time_s / n, 2),
            'avg_time_to_first_audio_ms': round(1000 * sum(g['ttfa_s'] for g in runs) / n),
            'inference_rate_audio_seconds_per_second': round(audio_s / time_s, 2),
            'real_time_factor': round(time_s / audio_s, 3),
            'minutes_to_generate_1_hour_of_audio': round(60 * time_s / audio_s, 1),
        },
        'memory': {'weights_gb': float(s.download_gb), 'peak_memory_gb': round(max(g['peak_gb'] for g in runs), 2)},
        **(HARDWARE.get(name) or {'hardware': 'not measured (macmon not available)'}),
        'energy': {
            'total_joules': round(energy, 1) if measured else None,
            'avg_joules_per_prompt': round(energy / n, 1) if measured else None,
            'per_audio_second_joules': round(energy / audio_s, 2) if measured else None,
            # J per audio second x 3,600 s = J per audio hour; / 3,600 = Wh per audio hour (the factors cancel)
            'per_audio_hour_watt_hours': round(energy / audio_s, 2) if measured else None,
        },
        'cost_usd': {
            'api_fee': 0.0,
            'electricity_whole_run': round(energy / 3.6e6 * ELECTRICITY_USD_PER_KWH, 7) if measured else None,
            # 1,000 audio hours = 3.6e6 audio seconds and 1 kWh = 3.6e6 J, so $ = J per audio second x price per kWh
            'electricity_per_1000_audio_hours': round(energy / audio_s * ELECTRICITY_USD_PER_KWH, 4) if measured else None,
            'electricity_price_per_kwh': ELECTRICITY_USD_PER_KWH,
        },
        'per_prompt': [{
            'id': g['id'], 'audio_seconds': round(g['audio_s'], 2), 'inference_seconds': round(g['time_s'], 3),
            'time_to_first_audio_ms': round(1000 * g['ttfa_s']), 'word_error_rate_pct': g['wer_pct'],
            'energy_joules': g['power']['energy_j'],
        } for g in runs],
    }

def finish_model(name, runs, model_dir, load_s):
    whisper_check(runs)
    csv_path = save_prompt_metrics(name, runs, model_dir)
    sm = simple_metrics(name, runs, load_s)
    (out_dir(name) / f'{name}_simple_metrics.json').write_text(json.dumps(sm, indent=2))
    print(f'Per-prompt metrics: {csv_path}')
    return sm

sm = finish_model(model_name, runs, model_dir, load_seconds)
rows = [
    ('Prompts / audio produced', f"{sm['prompts_run']} / {sm['audio_output']['total_seconds']} s"),
    ('Word error rate (Whisper round trip)', f"{sm['intelligibility']['word_error_rate_pct']}% ({sm['intelligibility']['perfect_prompts']} prompts perfect)"),
    ('Inference time (total)', f"{sm['speed']['total_inference_seconds']} s"),
    ('Inference rate', f"{sm['speed']['inference_rate_audio_seconds_per_second']} audio seconds per second"),
    ('Time to first audio (avg)', f"{sm['speed']['avg_time_to_first_audio_ms']} ms"),
    ('Peak memory', f"{sm['memory']['peak_memory_gb']} GB"),
    ('API cost', '$0.00 (runs locally)'),
]
if sm['energy']['total_joules'] is not None:
    rows += [
        ('GPU busy / power (avg)', f"{sm['gpu']['avg_utilization_pct']}% / {sm['gpu']['avg_power_watts']} W"),
        ('CPU busy / power (avg)', f"{sm['cpu']['avg_utilization_pct']}% / {sm['cpu']['avg_power_watts']} W"),
        ('NPU (Neural Engine) power (avg)', f"{sm['npu']['avg_power_watts']} W"),
        ('Energy per second of audio', f"{sm['energy']['per_audio_second_joules']} J"),
        (f'Electricity per 1,000 hours of audio (at ${ELECTRICITY_USD_PER_KWH}/kWh)', f"${sm['cost_usd']['electricity_per_1000_audio_hours']}"),
    ]
display(pd.DataFrame(rows, columns=['Metric', model_name]).style.hide(axis='index'))
pd.read_csv(Path(model_dir) / f'{model_name}_prompt_metrics.csv', dtype={'prompt_id': str}).drop(
    columns=['model', 'family', 'size', 'audio_file', 'prompt', 'description']).style.hide(axis='index')

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 2134.78it/s]

Per-prompt metrics: metrics_analysis/tts/Qwen3-TTS/1.7B/tts_prompts_described/Model_Level_Metrics/Qwen3-TTS-1.7B_prompt_metrics.csv


Metric,Qwen3-TTS-1.7B
Prompts / audio produced,10 / 126.5 s
Word error rate (Whisper round trip),0.37% (9 prompts perfect)
Inference time (total),29.97 s
Inference rate,4.22 audio seconds per second
Time to first audio (avg),2988 ms
Peak memory,10.1 GB
API cost,$0.00 (runs locally)
GPU busy / power (avg),93.1% / 11.5 W
CPU busy / power (avg),7.1% / 7.4 W
NPU (Neural Engine) power (avg),0.0 W


prompt_id,voice,description_used,input_words,input_characters,output_audio (s),sample_rate (Hz),time_to_first_audio (ms),inference_rate (audio s/s),inference_time (s),word_error_rate (%),transcript
01,ryan,full description,34,227,18.240000,24000,4288.100000,4.240000,4.300000,0.000000,"Welcome to our annual conference on AI advancements. Today, we gather here to celebrate the latest breakthroughs in artificial intelligence. Let us explore innovative technologies and pave the way for a smarter future together."
02,vivian,full description,22,149,10.880000,24000,2586.700000,4.190000,2.596000,0.000000,Congratulations! You've won the grand prize! It's your moment to shine! Celebrate this amazing achievement and share the joy with everyone around you!
03,ryan,full description,28,157,11.040000,24000,2608.100000,4.220000,2.616000,0.000000,"Please make your way to the nearest exit in an orderly manner. For your safety and convenience, follow the clearly marked signs and stay calm as you proceed."
04,vivian,full description,29,155,11.760000,24000,2776.500000,4.220000,2.785000,0.000000,It's going to be a beautiful day with sunshine and clear skies. Step outside and enjoy the warmth of the sun while making the most of this perfect weather.
05,ryan,full description,25,150,10.400000,24000,2448.400000,4.230000,2.456000,0.000000,Our sale ends tonight at midnight. Don't miss out on these exclusive deals. Act now to take advantage of these incredible savings before it's too late.
06,vivian,full description,26,155,11.760000,24000,2773.900000,4.230000,2.783000,0.000000,"Can you believe it? We made it to the finals! After all our hard work and dedication, this incredible achievement is something to truly celebrate together."
07,ryan,full description,31,189,17.120000,24000,4024.900000,4.240000,4.037000,0.000000,"In 1492, Christopher Columbus set sail across the Atlantic, embarking on a journey that would forever change the course of history. His voyage marked a new era of exploration and discovery."
08,vivian,full description,28,155,11.520000,24000,2764.000000,4.160000,2.772000,0.000000,Your food will be served shortly. Thank you for your patience as we prepare your meal with care and attention to ensure the best dining experience for you.
09,ryan,full description,23,143,13.840000,24000,3251.400000,4.240000,3.262000,4.350000,"The operation could not be completed. Please, try again later or contact support if the issue persists. Thank you for your understanding."
10,ryan,full description,25,155,9.920000,24000,2357.100000,4.190000,2.366000,0.000000,"Are you ready for an adventure of a lifetime? Pack your bags, gear up, and let's explore new horizons filled with excitement and unforgettable experiences."


## 8. Compare every size in the family

Runs the prompts through every other size of `FAMILY` (reusing the results already produced for `SIZE`), unloading each
model before the next. Set `TTS_COMPARE=0` (or `RUN_FAMILY_COMPARISON = False` in section 1) to skip.

In [9]:
import gc

results = {SIZE: (sm, runs)}
if RUN_FAMILY_COMPARISON:
    del model
    gc.collect()
    mx.clear_cache()
    for size, name in MODEL_NAME[PLATFORM][TASK][FAMILY].items():
        if size == SIZE:
            continue
        print(f'{size}: {name}')
        m, load_s = load_tts_model(name)
        r, mdir = run_prompt_set(m, name)
        del m
        gc.collect()
        mx.clear_cache()
        results[size] = (finish_model(name, r, mdir, load_s), r)

order = [s for s in ('SMALL', 'MEDIUM', 'LARGE') if s in results]
compare = pd.DataFrame({
    s: {
        'Model': r_['model'],
        'Params (M)': f"{r_['parameters_million']:,}",
        'Audio produced (s)': r_['audio_output']['total_seconds'],
        'Word error rate (%)': r_['intelligibility']['word_error_rate_pct'],
        'Time to first audio (ms)': r_['speed']['avg_time_to_first_audio_ms'],
        'Inference rate (audio s/s)': r_['speed']['inference_rate_audio_seconds_per_second'],
        'Inference time (s)': r_['speed']['total_inference_seconds'],
        'Peak memory (GB)': r_['memory']['peak_memory_gb'],
        'GPU busy (%)': r_.get('gpu', {}).get('avg_utilization_pct'),
        'GPU power (W)': r_.get('gpu', {}).get('avg_power_watts'),
        'Energy / audio second (J)': r_['energy']['per_audio_second_joules'],
    } for s in order for r_ in [results[s][0]]
})
compare.columns.name = FAMILY
compare

SMALL: Qwen3-TTS-0.6B


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 3342.74it/s]


[transformers] You are using a model of type `qwen3_tts` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


  Initialized encoder codebooks
Loaded speech tokenizer from /Users/genailab/.cache/huggingface/hub/models--mlx-community--Qwen3-TTS-12Hz-0.6B-CustomVoice-bf16/snapshots/6415d95f88be018ff9e46813119dc3bc12261328/speech_tokenizer


...

  [10]  25 words ->  19.1s audio in  3.59s ( 5.32 audio s/s, first audio   3582 ms)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 2136.95it/s]

Per-prompt metrics: metrics_analysis/tts/Qwen3-TTS/0.6B/tts_prompts_described/Model_Level_Metrics/Qwen3-TTS-0.6B_prompt_metrics.csv


Qwen3-TTS,SMALL,LARGE
Model,Qwen3-TTS-0.6B,Qwen3-TTS-1.7B
Params (M),"1,076","2,087"
Audio produced (s),143.4,126.5
Word error rate (%),0.37,0.37
Time to first audio (ms),2699,2988
Inference rate (audio s/s),5.3,4.22
Inference time (s),27.08,29.97
Peak memory (GB),8.58,10.1
GPU busy (%),91.1,93.1
GPU power (W),12.1,11.5


In [10]:
# Word error rate (%) per prompt and size
pd.DataFrame({s: {g['id']: g['wer_pct'] for g in results[s][1]} for s in order}).rename_axis('prompt id')

,SMALL,LARGE
prompt id,,
01,0.00,0.00
02,0.00,0.00
03,0.00,0.00
04,0.00,0.00
05,0.00,0.00
06,0.00,0.00
07,0.00,0.00
08,0.00,0.00
09,4.35,4.35


## 9. All models on this prompt set

Collects every model run so far (any family and size, from this notebook or terminal runs):
`outputs/tts/all_models_<prompt set>.json` / `.csv` from the simple metrics, and
`metrics_analysis/tts/all_models_prompt_metrics_<prompt set>.csv` from the per-prompt CSVs.

In [11]:
FAMILY_ORDER = list(MODEL_NAME[PLATFORM][TASK])
SIZE_ORDER = ['SMALL', 'MEDIUM', 'LARGE']
files = Path('outputs/tts').glob(f'*/*/{PROMPT_SET}/*_simple_metrics.json')
models = sorted((json.loads(f.read_text()) for f in files),
                key=lambda m: (FAMILY_ORDER.index(m['family']), SIZE_ORDER.index(m['size'])))
overview = [{k: v for k, v in m.items() if k != 'per_prompt'} for m in models]
all_json = Path('outputs/tts') / f'all_models_{PROMPT_SET}.json'
all_json.write_text(json.dumps({'prompt_set': PROMPT_SET, 'models': overview}, indent=2))
flat = pd.json_normalize(overview)
flat.to_csv(all_json.with_suffix('.csv'), index=False)
print(f'{len(models)} models -> {all_json} and {all_json.with_suffix(".csv")}')

pd.DataFrame({
    'Family': flat['family'], 'Size': flat['size'], 'Model': flat['model'],
    'Params (M)': flat['parameters_million'],
    'WER (%)': flat['intelligibility.word_error_rate_pct'],
    'Audio (s)': flat['audio_output.total_seconds'],
    'Audio s/s': flat['speed.inference_rate_audio_seconds_per_second'],
    'First audio (ms)': flat['speed.avg_time_to_first_audio_ms'],
    'Peak memory (GB)': flat['memory.peak_memory_gb'],
    'GPU busy (%)': flat.get('gpu.avg_utilization_pct'),
    'GPU power (W)': flat.get('gpu.avg_power_watts'),
    'CPU busy (%)': flat.get('cpu.avg_utilization_pct'),
    'NPU power (W)': flat.get('npu.avg_power_watts'),
    'J / audio second': flat['energy.per_audio_second_joules'],
}).style.hide(axis='index')

8 models -> outputs/tts/all_models_tts_prompts_described.json and outputs/tts/all_models_tts_prompts_described.csv


Family,Size,Model,Params (M),WER (%),Audio (s),Audio s/s,First audio (ms),Peak memory (GB),GPU busy (%),GPU power (W),CPU busy (%),NPU power (W),J / audio second
Bark,SMALL,bark-small,420,19.050000,168.400000,1.110000,1499,4.210000,75.300000,25.000000,8.300000,0.000000,33.080000
Bark,LARGE,bark,1144,5.860000,154.300000,0.350000,3756,7.250000,79.300000,20.600000,9.300000,0.000000,86.520000
Kitten-TTS,SMALL,kitten-tts-nano,15,2.930000,131.300000,93.740000,134,2.400000,51.400000,6.400000,8.800000,0.000000,0.140000
Kitten-TTS,MEDIUM,kitten-tts-micro,36,5.860000,135.700000,30.990000,417,1.880000,51.300000,5.200000,9.500000,0.000000,0.460000
Kitten-TTS,LARGE,kitten-tts-mini,74,3.300000,116.300000,26.300000,421,2.780000,59.400000,9.700000,9.300000,0.000000,0.740000
Qwen3-TTS,SMALL,Qwen3-TTS-0.6B,1076,0.370000,143.400000,5.300000,2699,8.580000,91.100000,12.100000,7.500000,0.000000,5.090000
Qwen3-TTS,LARGE,Qwen3-TTS-1.7B,2087,0.370000,126.500000,4.220000,2988,10.100000,93.100000,11.500000,7.100000,0.000000,6.430000
OuteTTS,MEDIUM,OuteTTS-1.0-1B,1248,0.000000,112.600000,0.920000,11171,7.690000,94.200000,7.500000,6.100000,0.000000,25.060000


In [12]:
# Every model's per-prompt metrics (Model_Level_Metrics) in one CSV
prompt_csvs = sorted(Path('metrics_analysis/tts').glob(f'*/*/{PROMPT_SET}/Model_Level_Metrics/*_prompt_metrics.csv'))
all_prompts = pd.concat([pd.read_csv(p, dtype={'prompt_id': str}) for p in prompt_csvs], ignore_index=True)
all_prompts['family'] = pd.Categorical(all_prompts['family'], FAMILY_ORDER, ordered=True)
all_prompts['size'] = pd.Categorical(all_prompts['size'], SIZE_ORDER, ordered=True)
all_prompts = all_prompts.sort_values(['family', 'size', 'prompt_id'])
combined_csv = Path('metrics_analysis/tts') / f'all_models_prompt_metrics_{PROMPT_SET}.csv'
all_prompts.to_csv(combined_csv, index=False)
print(f'{len(prompt_csvs)} models, {len(all_prompts)} prompt rows -> {combined_csv}')
all_prompts.groupby(['family', 'size', 'model'], observed=True).agg(
    prompts=('prompt_id', 'count'), audio_s=('output_audio (s)', 'sum'),
    avg_first_audio_ms=('time_to_first_audio (ms)', 'mean'), avg_rate=('inference_rate (audio s/s)', 'mean'),
    total_time_s=('inference_time (s)', 'sum'), avg_wer_pct=('word_error_rate (%)', 'mean')).round(2)

8 models, 80 prompt rows -> metrics_analysis/tts/all_models_prompt_metrics_tts_prompts_described.csv


prompts  audio_s  avg_first_audio_ms  \
family     size   model                                                    
Bark       SMALL  bark-small             10   168.42             1499.44   
           LARGE  bark                   10   154.32             3755.58   
Kitten-TTS SMALL  kitten-tts-nano        10   131.35              134.17   
           MEDIUM kitten-tts-micro       10   135.68              416.63   
           LARGE  kitten-tts-mini        10   116.34              420.64   
Qwen3-TTS  SMALL  Qwen3-TTS-0.6B         10   143.44             2698.84   
           LARGE  Qwen3-TTS-1.7B         10   126.48             2987.91   
OuteTTS    MEDIUM OuteTTS-1.0-1B         10   112.60            11170.62   

                                    avg_rate  total_time_s  avg_wer_pct  
family     size   model                                                  
Bark       SMALL  bark-small            1.10        152.25        18.98  
           LARGE  bark                  0.36        439.49         5.99  
Kitten-TTS SMALL  kitten-tts-nano      92.19          1.40         3.21  
           MEDIUM kitten-tts-micro     31.04          4.38         6.24  
           LARGE  kitten-tts-mini      26.07          4.42         3.61  
Qwen3-TTS  SMALL  Qwen3-TTS-0.6B        5.29         27.09         0.43  
           LARGE  Qwen3-TTS-1.7B        4.22         29.97         0.43  
OuteTTS    MEDIUM OuteTTS-1.0-1B        0.92        122.89         0.00